## 测试代码 接口的地方

In [1]:
from collections import *
import torch

c = Counter('banana')
c

Counter({'a': 3, 'n': 2, 'b': 1})

### BPE编码

#### Unicode编码相关测试

In [2]:
ord('牛'), chr(29275), ord('😋'), chr(128523)

(29275, '牛', 128523, '😋')

In [3]:
chr(0), print(chr(0))

 


('\x00', None)

In [4]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [5]:
print("this is a test" + chr(0) + "string")

this is a test string


In [6]:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")
print(utf8_encoded)
print(type(utf8_encoded))
print(list(utf8_encoded))
# 一个Byte并不一定和一个Unicode字符相关联，因为有些字符的utf-8编码是多个字节的
print(len(test_string))
print(len(utf8_encoded))
print(utf8_encoded.decode('utf-8'))

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
<class 'bytes'>
[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]
13
23
hello! こんにちは!


In [7]:
print(test_string.encode('utf-16'))
print(test_string.encode('utf-32'))

b'\xff\xfeh\x00e\x00l\x00l\x00o\x00!\x00 \x00S0\x930k0a0o0!\x00'
b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00!\x00\x00\x00 \x00\x00\x00S0\x00\x00\x930\x00\x00k0\x00\x00a0\x00\x00o0\x00\x00!\x00\x00\x00'


In [8]:
# 错误的逐字节解码函数，因为中文等字符不是一个字节对应一个字符，中文在utf8通常是3字节编码
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

decode_utf8_bytes_to_str_wrong('hello! 你好啊!'.encode('utf8'))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 0: unexpected end of data

#### 基于正则的预分词器

In [ ]:
# regex是对Unicode支持更完善的第三方正则包，re plus版
import regex as re

PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

print(re.findall(PAT, "some text that i'll pre-tokenize  你好世界，非常高兴认识你"))
# 使用finditer迭代器来防止储存大量预分词结果在内存
print(re.finditer(PAT, "some text that i'll pre-tokenize  你好世界，非常高兴认识你"))
print(re.escape("<|endoftext|>"))  # 给字符串里有正则特殊符号的字符加转义，比如把或|变成\|

print(re.findall(PAT, """u don't have to be scared of the loud dog, I'll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.
"""))

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize', ' ', ' 你好世界', '，', '非常高兴认识你']
<\|endoftext\|>
['u', ' don', "'t", ' have', ' to', ' be', ' scared', ' of', ' the', ' loud', ' dog', ',', ' I', "'ll", ' protect', ' you', '".', ' The', ' mole', ' felt', ' so', ' safe', ' with', ' the', ' little', ' girl', '.', ' She', ' was', ' very', ' kind', ' and', ' the', ' mole', ' soon', ' came', ' to', ' trust', ' her', '.', ' He', ' leaned', ' against', ' her', ' and', ' she', ' kept', ' him', ' safe', '.', ' The', ' mole', ' had', ' found', ' his', ' best', ' friend', '.', '\n']


Bad pipe message: %s [b'"Chromium";v="148", "Microsoft Edge";v="148", "Not/A)Bran']
Bad pipe message: %s [b';v="99"\r\nsec-ch-ua-mobile: ?0\r\nsec-ch-ua-platform: "Windows"\r\nUpgrade-Insecure-Requests: 1\r\nUser-Ag']
Bad pipe message: %s [b't: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0']
Bad pipe message: %s [b'afari/537.36 Edg/148.0.0.0\r\nAcc', b't: text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,']
Bad pipe message: %s [b'plication/signed-exchange;v=b3;q=0.7\r\nSec-Fetch-Site: none\r\nSec-Fetch-Mode: navigate\r\nSec-Fetch-']
Bad pipe message: %s [b'ol: max-age=0\r\nsec-ch-ua: "Chromium";v="148", "Microsoft Edge";v="148", "Not/A)Brand";v="99"\r\nsec-ch-ua-mobile: ?0\r']
Bad pipe message: %s [b'ec-ch-ua-', b'atform: "Windows"\r\nUpgrade-Insecure-Requests: 1\r\nUser-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWeb']
Bad pipe message: %s [b't/537.36 (KHTML, like Gecko) Chrom

In [ ]:
# 字典序比较排序，优先看第一个元素，BA大于B
max([("A", "B"), ("A", "C"), ("B", "ZZ"), ("BA", "A")])

('BA', 'A')

In [ ]:
tuple([1,2,3]), tuple('abc  defg'),   # tuple会直接把字符串逐char拆开

((1, 2, 3), ('a', 'b', 'c', ' ', ' ', 'd', 'e', 'f', 'g'))

In [ ]:
tuple(['abc', 'def']), ('a', 'b') == ('ab')

(('abc', 'def'), False)

In [ ]:
# int转bytes要放在可迭代对象里
bytes(5), bytes([5]), bytes([4, 5])

(b'\x00\x00\x00\x00\x00', b'\x05', b'\x04\x05')

In [ ]:
a = {1:0, 3:4, 4:0}
def big(x):
    print(type(x))
    return x

print(max(a.items(), key=big))  # max实际上是把每个元素扔给key的函数，然后在所有元素经过key处理后再比较，通常返回元组来比较

print(max(a, key=lambda x: a[x]))
for i in a.items():
    print(i[1])

<class 'tuple'>
<class 'tuple'>
<class 'tuple'>
(4, 0)
3
0
4
0


In [ ]:
'abc'.encode('utf-8')

b'abc'

In [ ]:
import regex as re
re.split(pattern='a', string='a22aa3a\na')
# 要特别注意这里re.split会切出空字符串

['', '22', '', '3', '\n', '']

In [ ]:
spec_tokens = ['12^|', '||']
print(re.escape(spec_tokens))  # 这里结果是错误的，可以用列表生成器来批量迭代转义，不用写大的for循环
[re.escape(token) for token in spec_tokens]

12^|||


['12\\^\\|', '\\|\\|']

In [3]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math
class SGD(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
            # print(f'group: {group}')
            for p in group["params"]:
                print(f'group["params"]:{group["params"]}')
                print(f'p: {p}')
                if p.grad is None:
                    continue
                state = self.state[p] # Get state associated with p.
                t = state.get("t", 0) # Get iteration number from the state, or 0.
                grad = p.grad.data # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
                state["t"] = t + 1 # Increment iteration number.
        return loss

weights = torch.nn.Parameter(5 * torch.randn((10, 10)))
opt = SGD([weights], lr=1000)
# print(opt.param_groups)

for t in range(10):
    opt.zero_grad()
    loss = (weights ** 2).mean()
    print(loss.cpu().item())
    loss.backward()
    opt.step()

28.845842361450195
group["params"]:[Parameter containing:
tensor([[ 3.4071e+00, -2.2085e+00, -4.9296e+00,  2.5678e+00, -1.8376e-01,
         -3.4552e+00, -8.5288e+00,  6.3815e+00,  4.6712e+00,  5.1262e+00],
        [ 3.5809e-01,  6.2310e+00, -4.8340e+00,  2.6084e+00, -1.5725e+00,
         -1.3606e+00,  4.7779e+00,  3.1118e+00,  1.5728e+00,  1.5800e+00],
        [ 1.1665e+01,  3.5573e-01,  5.1921e+00,  4.4412e+00, -6.1587e+00,
         -2.1617e+00, -1.2085e+01,  4.5849e+00,  1.7998e-03,  5.2193e+00],
        [ 1.1280e+01, -4.4170e-01, -1.1954e+00, -2.6441e+00, -3.2148e+00,
         -1.2395e+01,  1.7315e+00, -4.2000e+00,  1.1373e+01, -1.1832e+00],
        [ 2.0322e+00,  4.3337e+00, -3.6052e+00, -1.7349e+00, -1.6868e+00,
          4.1132e-01,  1.6544e+00, -6.8371e-01, -1.3723e+00, -2.3720e+00],
        [-3.4892e-01, -1.0409e+01,  7.0730e+00,  3.3860e+00, -2.5363e+00,
         -6.2297e-01,  4.3810e+00,  1.0952e+01, -3.9741e+00, -1.0158e+01],
        [ 7.5809e-01,  3.6389e-01,  6.8405e+00, 

In [10]:
import torch

a = torch.tensor(5.0, requires_grad=True)
y = a ** 2
y.backward()
a.grad, a.grad.requires_grad


(tensor(10.), False)